Assignemnt 1.1:
Write Python Behavioral code, verilog RTL code for a 4-bit Counter
Write testbench and simulate using iverilog, display waveforms using matplotlib

4-bit Counter

Python Behavioral code

In [ ]:
class Counter4Bit:
    """Behavioral model of a 4-bit up-counter with active-high reset."""
    def __init__(self):
        self.count = 0

    def trigger(self, rst: bool, clk_posedge: bool) -> int:
        if rst:
            self.count = 0
        elif clk_posedge:
            self.count = (self.count + 1) % 16
        return self.count

# Example Usage
ctr = Counter4Bit()
print("Initial count =", ctr.count)
print("Trigger clk (rst=False) -> Count =", ctr.trigger(False, True))
print("Trigger clk (rst=False) -> Count =", ctr.trigger(False, True))
print("Trigger rst (rst=True)  -> Count =", ctr.trigger(True, False))


Verilog RTL Code

Install iverilog

In [ ]:
# install iverilog
!apt-get update
!apt-get install iverilog

Write RTL counter4b.v

In [ ]:
%%writefile counter4b.v
module counter4b (
    input  wire       clk,
    input  wire       rst,
    output reg  [3:0] count
);
    always @(posedge clk or posedge rst) begin
        if (rst) begin
            count <= 4'b0000;
        end else begin
            count <= count + 1'b1;
        end
    end
endmodule

Write Testbench tb_counter4b.v

In [ ]:
%%writefile tb_counter4b.v
`timescale 1ns / 1ps

module tb_counter4b;
    reg clk;
    reg rst;
    wire [3:0] count;

    counter4b uut (
        .clk(clk),
        .rst(rst),
        .count(count)
    );

    // Clock generation (10ns period -> toggles every 5ns)
    always #5 clk = ~clk;

    initial begin
        // --- Added for Waveform Generation ---
        $dumpfile("counter_waveform.vcd");
        $dumpvars(0, tb_counter4b);
        // -------------------------------------

        $monitor("Time = %0d ns | Clk = %b | Rst = %b | Count = %d (%b)", $time, clk, rst, count, count);

        // Initialize system
        clk = 0;
        rst = 1;
        #12; // Release reset shortly after the first clock edge
        
        rst = 0;
        #50; // Let it count up for 5 cycles
        
        rst = 1; #10; // Pulsing reset to verify recovery override
        rst = 0; #30;

        $finish;
    end
endmodule


Simulate 4-bit Counter

In [ ]:
!iverilog -o counter_sim counter4b.v tb_counter4b.v
!vvp counter_sim


Visualize waveform.vcd file using Matplotlib

In [ ]:
import matplotlib.pyplot as plt

# Sample custom time-ticks reflecting transitions up to simulation end
time_ticks = [0, 5, 10, 12, 15, 20, 25, 30, 35, 40, 45, 50, 55, 60, 62, 65, 70, 75, 80, 85, 90, 100]

# Manual timeline track mappings based on the testbench trace vector output logs
signal_Clk = [0, 1,  0,  0,  1,  0,  1,  0,  1,  0,  1,  0,  1,  0,  0,  1,  0,  1,  0,  1,  0,   0]
signal_Rst = [1, 1,  1,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  1,  1,  1,  0,  0,  0,  0,   0]
signal_Cnt = [0, 0,  0,  0,  1,  1,  2,  2,  3,  3,  4,  4,  5,  5,  0,  0,  0,  0,  1,  1,  2,   2]

# Create the digital waveform viewer diagram layout rows
fig, step_plots = plt.subplots(3, 1, figsize=(8, 6), sharex=True)
fig.suptitle("4-Bit Synchronous Up-Counter Signal Timing Waveform", fontsize=13, fontweight='bold')

signals = [("Clock", signal_Clk), ("Reset", signal_Rst), ("Count[3:0]", signal_Cnt)]
colors = ['#d62728', '#ff7f0e', '#9467bd']

for idx, (name, data) in enumerate(signals):
    step_plots[idx].step(time_ticks, data, where='post', color=colors[idx], linewidth=2.5)
    step_plots[idx].set_ylabel(name, fontsize=11, fontweight='bold', rotation=0, labelpad=35)
    
    if name == "Count[3:0]":
        step_plots[idx].set_ylim(-0.5, 6.5)
        step_plots[idx].set_yticks(range(6))
    else:
        step_plots[idx].set_ylim(-0.2, 1.2)
        step_plots[idx].set_yticks([0, 1])
        
    step_plots[idx].grid(True, which='both', linestyle=':', alpha=0.6)

plt.xlabel("Simulation Time (ns)", fontsize=11)
plt.xlim(0, 102)
plt.tight_layout()
plt.show()
